# 03 — Double-Sérsic mock with CompositeModel

This notebook validates the first multi-component BayMorph workflow.

Workflow:

1. build a bulge component;
2. build a disk component;
3. combine them with `CompositeModel`;
4. recover the two component fluxes with `DynestyFitter`;
5. repeat the experiment with Gaussian noise and PSF convolution using `ConvolvedModel`.

The goal is architectural validation: `CompositeModel`, `ConvolvedModel`, `ParameterSet`, `DynestyFitter`, and `PosteriorResult` should work together without changing the fitting backend.

## Imports

In [ ]:
import matplotlib.pyplot as plt

from baymorph import (
    CompositeModel,
    ConvolvedModel,
    DynestyFitter,
    GaussianPSF,
    Parameter,
    ParameterSet,
    SersicModel,
    add_gaussian_noise,
    make_constant_noise_map,
    plot_data_model_residual,
)

In [ ]:
from matplotlib.colors import LogNorm

BAYMORPH_IMAGE_CMAP = "turbo"


def _log_limits(
    *images,
    lower_percentile=0.5,
    upper_percentile=99.9,
    floor=1e-3,
):
    """Return robust positive limits for logarithmic image display."""
    positive_values = []

    for image in images:
        values = np.asarray(
            image,
            dtype=float,
        )
        values = values[
            np.isfinite(values)
            & (values > 0)
        ]

        if values.size > 0:
            positive_values.append(values)

    if len(positive_values) == 0:
        return floor, 1.0

    positive_values = np.concatenate(
        positive_values,
    )

    vmin = np.percentile(
        positive_values,
        lower_percentile,
    )
    vmax = np.percentile(
        positive_values,
        upper_percentile,
    )

    vmin = max(
        float(vmin),
        floor,
    )
    vmax = max(
        float(vmax),
        vmin * 10,
    )

    return vmin, vmax


def show_log_image(
    image,
    title,
    cmap=BAYMORPH_IMAGE_CMAP,
    colorbar_label="Flux",
):
    """Display an image using a robust logarithmic normalization."""
    vmin, vmax = _log_limits(
        image,
    )

    display_image = np.clip(
        np.asarray(
            image,
            dtype=float,
        ),
        vmin,
        vmax,
    )

    plt.figure(
        figsize=(6, 6),
    )

    plt.imshow(
        display_image,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )

    plt.colorbar(
        label=colorbar_label,
    )

    plt.title(
        f"{title} (log scale)",
    )

    plt.show()


def plot_log_data_model_residual(
    data,
    model,
    title="",
    cmap=BAYMORPH_IMAGE_CMAP,
):
    """Plot data and model on a log scale, with a linear residual panel."""
    data = np.asarray(
        data,
        dtype=float,
    )

    model = np.asarray(
        model,
        dtype=float,
    )

    residual = data - model

    vmin, vmax = _log_limits(
        data,
        model,
    )

    data_display = np.clip(
        data,
        vmin,
        vmax,
    )

    model_display = np.clip(
        model,
        vmin,
        vmax,
    )

    residual_limit = np.nanpercentile(
        np.abs(residual),
        99,
    )

    if not np.isfinite(residual_limit) or residual_limit == 0:
        residual_limit = 1.0

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4),
    )

    data_im = axes[0].imshow(
        data_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[0].set_title(
        "Data (log)"
    )
    fig.colorbar(
        data_im,
        ax=axes[0],
    )

    model_im = axes[1].imshow(
        model_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[1].set_title(
        "Model (log)"
    )
    fig.colorbar(
        model_im,
        ax=axes[1],
    )

    residual_im = axes[2].imshow(
        residual,
        origin="lower",
        cmap="RdBu_r",
        vmin=-residual_limit,
        vmax=residual_limit,
    )
    axes[2].set_title(
        "Residual"
    )
    fig.colorbar(
        residual_im,
        ax=axes[2],
    )

    if title:
        fig.suptitle(
            title,
            y=1.02,
        )

    plt.tight_layout()
    plt.show()


## True component parameters

In [ ]:
true_flux_bulge = 3000.0
true_flux_disk = 7000.0

shared_geometry = {
    "q": 0.8,
    "theta": 0.0,
    "x0": 30.0,
    "y0": 30.0,
}

bulge_shape = {
    "re": 3.0,
    "n": 4.0,
}

disk_shape = {
    "re": 12.0,
    "n": 1.0,
}

## Build true CompositeModel

In [ ]:
bulge = SersicModel(
    flux=true_flux_bulge,
    **bulge_shape,
    **shared_geometry,
)

disk = SersicModel(
    flux=true_flux_disk,
    **disk_shape,
    **shared_geometry,
)

true_model = CompositeModel(
    [
        bulge,
        disk,
    ]
)

## Render clean composite mock

In [ ]:
data = true_model.render(
    shape=(60, 60),
)

noise = make_constant_noise_map(
    data.shape,
    sigma=1.0,
)

In [ ]:
show_log_image(
    data,
    title="Clean double-Sérsic mock",
)

print("Total flux:", data.sum())
print("True bulge flux:", true_flux_bulge)
print("True disk flux:", true_flux_disk)

## Clean model factory

In [ ]:
def model_factory(
    flux_bulge,
    flux_disk,
):
    bulge = SersicModel(
        flux=flux_bulge,
        **bulge_shape,
        **shared_geometry,
    )

    disk = SersicModel(
        flux=flux_disk,
        **disk_shape,
        **shared_geometry,
    )

    return CompositeModel(
        [
            bulge,
            disk,
        ]
    )

## Parameters

In [ ]:
parameters = ParameterSet(
    [
        Parameter(
            "flux_bulge",
            500.0,
            6000.0,
        ),
        Parameter(
            "flux_disk",
            3000.0,
            12000.0,
        ),
    ]
)

## Fit clean mock

In [ ]:
fitter = DynestyFitter(
    model_factory=model_factory,
    data=data,
    noise=noise,
    parameters=parameters,
)

result = fitter.run(
    nlive=100,
    sample="rwalk",
    print_progress=True,
)

## Clean recovery

In [ ]:
best = result.best_parameters
best

In [ ]:
print(result.summary("flux_bulge"))
print(result.summary("flux_disk"))

In [ ]:
plot_log_data_model_residual(
    data,
    best_image,
    title="Clean double-Sérsic best fit",
)

## Add PSF convolution and noise

In [ ]:
psf = GaussianPSF(
    fwhm=3.0,
)

psf_true_model = ConvolvedModel(
    true_model,
    psf,
)

clean_psf_data = psf_true_model.render(
    shape=data.shape,
)

noisy_psf_data = add_gaussian_noise(
    clean_psf_data,
    sigma=1.0,
    seed=42,
)

noise_psf = make_constant_noise_map(
    noisy_psf_data.shape,
    sigma=1.0,
)

In [ ]:
show_log_image(
    noisy_psf_data,
    title="Noisy PSF-convolved double-Sérsic mock",
)

## PSF-convolved model factory

In [ ]:
def psf_model_factory(
    flux_bulge,
    flux_disk,
):
    composite = model_factory(
        flux_bulge=flux_bulge,
        flux_disk=flux_disk,
    )

    return ConvolvedModel(
        composite,
        psf,
    )

## Fit noisy PSF-convolved mock

In [ ]:
psf_fitter = DynestyFitter(
    model_factory=psf_model_factory,
    data=noisy_psf_data,
    noise=noise_psf,
    parameters=parameters,
)

psf_result = psf_fitter.run(
    nlive=150,
    sample="rwalk",
    print_progress=True,
)

## PSF recovery

In [ ]:
psf_best = psf_result.best_parameters
psf_best

In [ ]:
print(psf_result.summary("flux_bulge"))
print(psf_result.summary("flux_disk"))

In [ ]:
plot_log_data_model_residual(
    noisy_psf_data,
    psf_best_image,
    title="Noisy PSF-convolved double-Sérsic best fit",
)

## Quantitative recovery summary

In [ ]:
recovered_flux_bulge = psf_best["flux_bulge"]
recovered_flux_disk = psf_best["flux_disk"]

bulge_error_percent = 100 * (
    recovered_flux_bulge - true_flux_bulge
) / true_flux_bulge

disk_error_percent = 100 * (
    recovered_flux_disk - true_flux_disk
) / true_flux_disk

summary = {
    "parameter": [
        "flux_bulge",
        "flux_disk",
    ],
    "true": [
        true_flux_bulge,
        true_flux_disk,
    ],
    "recovered": [
        recovered_flux_bulge,
        recovered_flux_disk,
    ],
    "error_percent": [
        bulge_error_percent,
        disk_error_percent,
    ],
}

summary

In [ ]:
print("Notebook 03 conclusion:")
print("CompositeModel successfully represents a two-component Sérsic galaxy.")
print("DynestyFitter with ParameterSet recovers the two component fluxes.")
print("ConvolvedModel successfully applies PSF convolution to a CompositeModel.")
print("The noisy PSF-convolved residuals are dominated by injected Gaussian noise.")